# Lab Activity 3: Visualizing Functions with Contour Plots
**Course:** CSE473: Computational Intelligence — Mechatronics Engineering and Automation Program
**Prepares you for:** Lab Assignment 03 — Contour Plots with Matplotlib

## 🎯 Learning objectives
By the end of this lab you will be able to:
- Build evaluation grids with `np.meshgrid` and evaluate functions **element-wise** on them
- Draw filled contour plots with `ax.contourf`, overlay contour lines and a colorbar
- Locate minima and maxima on a grid with `argmin`/`argmax` + `np.unravel_index`
- Handle **singular** functions ($G_1$: weighted inverse distances) by masking
- Mark local extrema on a contour plot with `'x'` / `'*'` markers — the assignment's deliverable

⏱ **Estimated time: ~40–45 minutes**

## How this lab works
- The lab is split into **Parts**; each Part teaches one topic.
- Each Part starts with a short explanation plus a runnable **✏️ Worked example** — run it, tweak it, break it.
- Then you solve **🎯 Problems**. Read the task, write your code in the starter cell, and try it before opening any hints.
- Stuck? Open the **💡 Hint** blocks below each problem — Hint 1 is a nudge, Hint 2 names the approach. They get more specific as you go.
- Verify yourself with the **🧪 Self-check** cells — they run deterministic checks and fail with guiding messages until your solution is right.
- Truly stuck? The **✅ Reveal solution** block at the bottom of each hint section shows full working code.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("Setup OK — NumPy and matplotlib are ready.")

## Part 1: Grids with meshgrid & evaluating functions (≈10 min)

A contour plot shows a function $f(x, y)$ as colored bands over the plane. To draw it you first need the function's **values on a regular grid**.

- `np.meshgrid(x_vec, y_vec)` returns two 2-D arrays `X, Y` of shape `(len(y_vec), len(x_vec))`: `X` is **constant down each column** (it holds the x vector on every row), `Y` is **constant along each row** (it holds the y vector down every column).
- Write functions **element-wise** (`np.sin(X)`, `X**2`, ...) so the same code works on scalars and whole grids.
- Grid extrema: `Z.min()` / `Z.max()` give the values; `np.unravel_index(Z.argmin(), Z.shape)` gives the **(row, col)** location — remember `row → y`, `col → x`!

In [ ]:
# --- Worked example: build a grid, evaluate a bowl, find its minimum ---
x = np.linspace(-1, 8, 400)
y = np.linspace(-1, 8, 400)
X, Y = np.meshgrid(x, y)                 # both shape (400, 400)

print("grid shape:", X.shape)
print("X constant down a column:", np.allclose(X[1:, 7], X[:-1, 7]))
print("X varies along a row:", not np.allclose(X[7, 1:], X[7, :-1]))
print("Y constant along a row:", np.allclose(Y[7, 1:], Y[7, :-1]))

def bowl(x_, y_):                        # paraboloid with minimum at (2, 3)
    return (x_ - 2) ** 2 + (y_ - 3) ** 2

Z = bowl(X, Y)                           # element-wise over the whole grid
print("Z shape:", Z.shape, "| smallest value:", Z.min())

row, col = np.unravel_index(Z.argmin(), Z.shape)   # LOCATION of the minimum
print("minimum at (x, y) =", (x[col], y[row]), " <- col indexes x, row indexes y")

## 🎯 Problem 3.1 — Evaluate a function on a grid

**Given:** an element-wise function `func(x, y)`, axis limits `xlim`/`ylim` and a grid size `n`.

**Required:** write `evaluate_on_grid(func, xlim=(-1, 8), ylim=(-1, 8), n=400)` returning `(X, Y, Z)` where `X, Y` are the meshgrid arrays and `Z = func(X, Y)`, all of shape `(n, n)`.

**Expected output:** for `func = lambda a, b: a + 2*b` the Z array equals `X + 2*Y` element-wise. This helper powers every contour in this lab — and in **Lab Assignment 03**.

In [ ]:
def evaluate_on_grid(func, xlim=(-1, 8), ylim=(-1, 8), n=400):
    """Evaluate func(x, y) on an n x n grid over the given limits.

    Args:
        func: element-wise function of two arrays
        xlim: (xmin, xmax) limits for the x axis
        ylim: (ymin, ymax) limits for the y axis
        n: number of grid points per axis

    Returns:
        (X, Y, Z): meshgrid arrays and func evaluated on them, each shape (n, n)
    """
    # TODO: Your code here
    pass

# Demo call
demo = evaluate_on_grid(lambda a, b: a ** 2 + b ** 2, n=50)
if demo is not None:
    Xd, Yd, Zd = demo
    print("grid shapes:", Xd.shape, Yd.shape, Zd.shape)
    print("Z min:", Zd.min(), "(expected near 0 at the origin)")
else:
    print("Implement evaluate_on_grid to power this demo.")

<details>
<summary>💡 Hint 1 — three steps</summary>

You need 1-D coordinate vectors spanning the limits, two meshgrid arrays from them, and then the function called on those arrays. The worked example in Part 1 shows each piece.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
x = np.linspace(xlim[0], xlim[1], n)
y = np.linspace(ylim[0], ylim[1], n)
X, Y = np.meshgrid(x, y)
Z = func(X, Y)
return X, Y, Z
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def evaluate_on_grid(func, xlim=(-1, 8), ylim=(-1, 8), n=400):
    """Evaluate func(x, y) on an n x n grid over the given limits."""
    x = np.linspace(xlim[0], xlim[1], n)
    y = np.linspace(ylim[0], ylim[1], n)
    X, Y = np.meshgrid(x, y)
    Z = func(X, Y)
    return X, Y, Z
```
</details>

In [ ]:
# 🧪 Self-check for Problem 3.1
res = evaluate_on_grid(lambda a, b: a + 2 * b, n=40)
assert res is not None, "❌ evaluate_on_grid returned None — replace the 'pass'. See Hint 2 in Part 1."
Xg, Yg, Zg = res
assert Xg.shape == Yg.shape == Zg.shape == (40, 40), f"❌ Expected three (40, 40) arrays, got {Xg.shape}, {Yg.shape}, {Zg.shape}."
assert np.allclose(Xg[1:, 5], Xg[:-1, 5]), "❌ X should be constant down each column — check the order of the np.meshgrid arguments."
assert np.allclose(Yg[5, 1:], Yg[5, :-1]), "❌ Y should be constant along each row — check the order of the np.meshgrid arguments."
assert Xg.min() == -1 and Xg.max() == 8 and Yg.min() == -1 and Yg.max() == 8, "❌ The grid must span xlim/ylim exactly (np.linspace includes both endpoints)."
assert np.allclose(Zg, Xg + 2 * Yg), "❌ Z must equal func(X, Y) element-wise — call func on the meshgrid arrays, not on the 1-D vectors."
assert not np.allclose(Zg, Yg + 2 * Xg), "❌ Looks like x and y are swapped."
print("✅ Problem 3.1 passed — you can evaluate any function on a grid.")

## 🎯 Problem 3.2 — F(x, y) = sin²x + cos²y and its extrema

**Given:** nothing to load — you will implement the assignment's first function plus a grid-extrema locator.

**Required:** write two functions:

1. `F(x, y)` — the element-wise function $F(x, y) = \sin(x)^2 + \cos(y)^2$ (use `np.sin`, `np.cos`),
2. `locate_extrema(Z, x, y)` — taking a 2-D grid `Z` **and its 1-D coordinate vectors** `x`, `y`, returning `((x_min, y_min), (x_max, y_max))`: the coordinates of the smallest and largest grid values.

**Expected output:** on $[-1, 8]^2$, $F$ attains values in $[0, 2]$; the minimum sits where $\sin(x) = 0$ and $\cos(y) = 0$, the maximum where both squared terms equal 1.

In [ ]:
def F(x, y):
    """F(x, y) = sin(x)^2 + cos(y)^2, element-wise."""
    # TODO: Your code here
    pass

def locate_extrema(Z, x, y):
    """Coordinates of the minimum and maximum of grid values Z.

    Args:
        Z: 2-D array of function values on a grid
        x: 1-D x coordinates (grid columns)
        y: 1-D y coordinates (grid rows)

    Returns:
        ((x_min, y_min), (x_max, y_max))
    """
    # TODO: Your code here
    pass

# Demo call
demo_grid = evaluate_on_grid(F, n=200)
if demo_grid is not None:
    XF, YF, ZF = demo_grid
    ext = locate_extrema(ZF, XF[0, :], YF[:, 0])
    if ext is not None:
        print("F min at", ext[0], "| F max at", ext[1])
    else:
        print("Implement locate_extrema to see the extrema.")
else:
    print("Finish Problem 3.1 first to power this demo.")

<details>
<summary>💡 Hint 1 — mind the mapping</summary>

`argmin` gives a FLAT index. Convert it with `np.unravel_index(..., Z.shape)` to `(row, col)`; then the x coordinate comes from the COLUMN index and the y coordinate from the ROW index.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
def F(x, y):
    return np.sin(x) ** 2 + np.cos(y) ** 2

def locate_extrema(Z, x, y):
    r, c = np.unravel_index(np.argmin(Z), Z.shape)
    r2, c2 = np.unravel_index(np.argmax(Z), Z.shape)
    return (x[c], y[r]), (x[c2], y[r2])
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def F(x, y):
    """F(x, y) = sin(x)^2 + cos(y)^2, element-wise."""
    return np.sin(x) ** 2 + np.cos(y) ** 2

def locate_extrema(Z, x, y):
    """Coordinates of the minimum and maximum of grid values Z."""
    r_min, c_min = np.unravel_index(np.argmin(Z), Z.shape)
    r_max, c_max = np.unravel_index(np.argmax(Z), Z.shape)
    return (float(x[c_min]), float(y[r_min])), (float(x[c_max]), float(y[r_max]))
```
</details>

In [ ]:
# 🧪 Self-check for Problem 3.2
f00 = F(0.0, 0.0)
assert f00 is not None, "❌ F returned None — replace the 'pass'. See Hint 2 in Part 1."
assert abs(float(f00) - 1.0) < 1e-12, "❌ F(0, 0) = sin(0)^2 + cos(0)^2 = 1 — check your formula."
assert abs(float(F(1.0, 2.0)) - float(F(1.0 + np.pi, 2.0))) < 1e-12, "❌ F is periodic in x with period pi (sin^2) — check the first term."
assert abs(float(F(1.0, 2.0)) - float(F(1.0, 2.0 + np.pi))) < 1e-12, "❌ F is periodic in y with period pi (cos^2) — check the second term."
res2 = evaluate_on_grid(F, n=200)
assert res2 is not None, "❌ evaluate_on_grid returned None — finish Problem 3.1 first."
XF, YF, ZF = res2
assert ZF.min() >= -1e-9 and ZF.max() <= 2.0 + 1e-9, "❌ F must stay within [0, 2] — a square of sin/cos is never negative or above 1."
assert ZF.min() < 0.05, "❌ The grid minimum of F should be essentially 0 (sin = 0 and cos = 0 are reachable on the grid)."
assert ZF.max() > 1.95, "❌ The grid maximum of F should be essentially 2 (sin^2 = 1 and cos^2 = 1 are reachable on the grid)."
ext = locate_extrema(ZF, XF[0, :], YF[:, 0])
assert ext is not None, "❌ locate_extrema returned None — replace the 'pass'. See Hint 2 in Part 1."
min_pt, max_pt = ext
assert -1 <= min_pt[0] <= 8 and -1 <= min_pt[1] <= 8, "❌ The min point coordinates must lie inside the domain."
r_min, c_min = np.unravel_index(np.argmin(ZF), ZF.shape)
assert abs(min_pt[0] - XF[0, c_min]) < 1e-12 and abs(min_pt[1] - YF[r_min, 0]) < 1e-12, "❌ The min point must be the argmin location mapped back through x[col], y[row] — check the row/col -> y/x mapping."
assert float(F(*min_pt)) < 0.05 and float(F(*max_pt)) > 1.95, "❌ The reported points must actually attain the extreme F values."
print("✅ Problem 3.2 passed — F is tamed and its extrema are located.")

## Part 2: Your first contour plot (≈10 min)

With values on a grid, the plot itself is three calls:

- `ax.contourf(X, Y, Z, levels=15, cmap="viridis")` — filled color bands (returns a handle for the colorbar),
- `ax.contour(X, Y, Z, levels=15, colors="k", linewidths=0.3)` — thin band-boundary lines,
- `fig.colorbar(cs, ax=ax)` — the color scale.

Mark extrema with `ax.plot(px, py, "wx", ...)` for minima and `"w*"` for maxima (white markers read well on `viridis`). A helper that takes the values + extrema and **returns `(fig, ax)`** is exactly what **Lab Assignment 03** asks you to build.

In [ ]:
# --- Worked example: contour the bowl from Part 1 ---
fig, ax = plt.subplots(figsize=(6, 5))

cs = ax.contourf(X, Y, Z, levels=15, cmap="viridis")     # 15 filled bands
ax.contour(X, Y, Z, levels=15, colors="k", linewidths=0.3, alpha=0.4)
fig.colorbar(cs, ax=ax)                                   # color scale

row, col = np.unravel_index(Z.argmin(), Z.shape)
ax.plot(x[col], y[row], "wx", markersize=12, markeredgewidth=3, label="local min")
ax.plot(8, 8, "w*", markersize=14, label="largest grid value")
ax.legend(loc="upper right")
ax.set_title("bowl(x, y) = (x-2)^2 + (y-3)^2")
ax.set_xlabel("x"); ax.set_ylabel("y")
fig.tight_layout()
plt.show()

## 🎯 Problem 3.3 — The contour plot helper

**Given:** grid arrays `X, Y`, function values `values` of the same shape, a `title`, iterables of `min_points` / `max_points` (each an `(x, y)` tuple), and a `levels` count.

**Required:** write `plot_contour(X, Y, values, title, min_points=(), max_points=(), levels=15)` that draws the filled contour, the lines, the colorbar, plots each minimum with a white `'x'` and each maximum with a white `'*'`, labels the axes, and **returns `(fig, ax)`**.

**Expected output:** a matplotlib Figure with one contour axes (plus its colorbar), one white `'x'` per min point and one white `'*'` per max point. Lab Assignment 03 ships this helper as its Part 2 — building it here means the assignment's plotting is already done.

In [ ]:
def plot_contour(X, Y, values, title, min_points=(), max_points=(), levels=15):
    """Draw a filled contour plot of `values` over (X, Y) and mark extrema.

    Args:
        X, Y: meshgrid arrays
        values: 2-D function values on the grid
        title: plot title
        min_points: iterable of (x, y) minima to mark with 'x'
        max_points: iterable of (x, y) maxima to mark with '*'
        levels: number of contour levels

    Returns:
        (fig, ax): the matplotlib Figure and Axes
    """
    # TODO: Your code here
    pass

# Demo call (uses the bowl grid from the Part 1 worked example)
demo_out = plot_contour(X, Y, Z, "bowl demo",
                        min_points=[(2.0, 3.0)], max_points=[(8.0, 8.0)])
if demo_out is not None:
    fig_d, ax_d = demo_out
    print("plot_contour returned a", type(fig_d).__name__)
    plt.show()
else:
    print("Implement plot_contour to power this demo.")

<details>
<summary>💡 Hint 1 — copy the worked example</summary>

Everything you need is in this Part's worked example — turn it into a function that takes the values, a title and the extrema, and returns `(fig, ax)` so callers can build on it.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
fig, ax = plt.subplots(figsize=(6, 5))
cs = ax.contourf(X, Y, values, levels=levels, cmap="viridis")
ax.contour(X, Y, values, levels=levels, colors="k", linewidths=0.3)
fig.colorbar(cs, ax=ax)
for (px, py) in min_points: ax.plot(px, py, "wx", markersize=12)
for (px, py) in max_points: ax.plot(px, py, "w*", markersize=14)
ax.set_title(title)
return fig, ax
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def plot_contour(X, Y, values, title, min_points=(), max_points=(), levels=15):
    """Draw a filled contour plot of `values` over (X, Y) and mark extrema."""
    fig, ax = plt.subplots(figsize=(6, 5))
    cs = ax.contourf(X, Y, values, levels=levels, cmap="viridis")
    ax.contour(X, Y, values, levels=levels, colors="k", linewidths=0.3, alpha=0.4)
    fig.colorbar(cs, ax=ax)
    for (px, py) in min_points:
        ax.plot(px, py, "wx", markersize=12, markeredgewidth=3, label="local min")
    for (px, py) in max_points:
        ax.plot(px, py, "w*", markersize=14, label="local max")
    if min_points or max_points:
        ax.legend(loc="upper right")
    ax.set_title(title)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    fig.tight_layout()
    return fig, ax
```
</details>

In [ ]:
# 🧪 Self-check for Problem 3.3
import matplotlib.figure as _mpl_figure
import matplotlib.collections as _mpl_coll
from matplotlib.contour import QuadContourSet as _QCS

def _has_contourf(ax):
    # matplotlib >= 3.8 registers contourf as a QuadContourSet in ax.collections;
    # older versions register a QuadMesh. Accept either.
    return any(isinstance(c, (_mpl_coll.QuadMesh, _QCS)) for c in ax.collections)

xs = np.linspace(-2, 2, 30); ys = np.linspace(-2, 2, 30)
XX, YY = np.meshgrid(xs, ys)
ZZ = XX ** 2 + YY ** 2
out3 = plot_contour(XX, YY, ZZ, "test bowl", min_points=[(0.0, 0.0)], max_points=[(1.5, 1.5)])
assert out3 is not None, "❌ plot_contour returned None — replace the 'pass'. See Hint 2 in Part 2."
fig3, ax3 = out3
assert isinstance(fig3, _mpl_figure.Figure), "❌ Return (fig, ax) — fig must be a matplotlib Figure."
assert len(fig3.get_axes()) >= 2, "❌ Expected a contour axes plus a colorbar — did you call fig.colorbar(cs, ax=ax)?"
assert _has_contourf(ax3), "❌ No filled contour found — did you use ax.contourf (with an f)?"
assert len(ax3.get_lines()) == 2, f"❌ Expected exactly 2 marker lines (one per extreme), found {len(ax3.get_lines())}."
assert ax3.get_title() == "test bowl", "❌ The title must be set with ax.set_title(...)."
empty3 = plot_contour(XX, YY, ZZ, "no markers")
assert empty3 is not None and len(empty3[1].get_lines()) == 0, "❌ With no extrema given there should be no marker lines."
print("✅ Problem 3.3 passed — you own the contour helper.")

## Part 3: G1 — when a function blows up (≈10 min)

The assignment's second function is a sum of **weighted inverse distances** to three centers $(2, 4)$, $(5, 2)$, $(4, 5)$:

$$G_1(x, y) = \frac{9}{r_1} + \frac{12}{r_2} + \frac{25}{r_3}, \qquad r_i = \text{distance to center } i$$

- At each center $r_i \to 0$, so $G_1 \to +\infty$: the grid values **explode** in three tiny neighborhoods.
- The strongest weight (25 at $(4, 5)$) dominates the maximum.
- Strategy: evaluate with `np.errstate(divide="ignore")`, then **mask** every cell that is non-finite or above a `cap` (`np.ma.masked_where`) so `contourf` skips the singular cores. The minimum hides at the far corner of the domain.

In [ ]:
# --- Worked example: G1 blows up at its centers — mask it ---
def G1_example(x, y):
    return (9 / np.sqrt((x - 2) ** 2 + (y - 4) ** 2)
            + 12 / np.sqrt((x - 5) ** 2 + (y - 2) ** 2)
            + 25 / np.sqrt((x - 4) ** 2 + (y - 5) ** 2))

print("G1 near the (4, 5) center:", round(G1_example(4.0, 4.9), 1), " <- the 25/distance term explodes")
print("G1 far from all centers:", round(G1_example(-1.0, -1.0), 2))

with np.errstate(divide="ignore", invalid="ignore"):
    Z_g1 = G1_example(X, Y)
print("finite fraction on the grid:", round(float(np.isfinite(Z_g1).mean()), 4))

cap = 200.0
Z_g1_masked = np.ma.masked_where(~np.isfinite(Z_g1) | (Z_g1 > cap), Z_g1)
print("masked cells:", int(Z_g1_masked.mask.sum()), "of", Z_g1.size)
row, col = np.unravel_index(np.ma.argmin(Z_g1_masked), Z_g1_masked.shape)
print("masked grid minimum at (x, y) =", (x[col], y[row]), " <- the far corner")

## 🎯 Problem 3.4 — Implement G1

**Given:** the formula above (centers $(2, 4)$, $(5, 2)$, $(4, 5)$ with weights $9, 12, 25$).

**Required:** write `G1(x, y)` computing the weighted sum of inverse distances, **element-wise** (`np.sqrt` of the squared distances). No masking here — return the raw values.

**Expected output:** positive everywhere it is finite; enormous near $(4, 5)$; decreasing as you move away from a center along a fixed ray.

In [ ]:
def G1(x, y):
    """G1(x, y) = 9/sqrt((x-2)^2+(y-4)^2) + 12/sqrt((x-5)^2+(y-2)^2)
                  + 25/sqrt((x-4)^2+(y-5)^2), element-wise."""
    # TODO: Your code here
    pass

# Demo call
demo_v = G1(4.0, 4.9)
if demo_v is not None:
    print("G1(4.0, 4.9) =", demo_v)
    print("G1(-1.0, -1.0) =", G1(-1.0, -1.0))
else:
    print("Implement G1 to power this demo.")

<details>
<summary>💡 Hint 1 — one term per center</summary>

Three terms of the same shape: weight divided by `np.sqrt` of the squared distance. The third term uses weight 25 and center (4, 5) — that one dominates.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
return (9 / np.sqrt((x - 2) ** 2 + (y - 4) ** 2)
        + 12 / np.sqrt((x - 5) ** 2 + (y - 2) ** 2)
        + 25 / np.sqrt((x - 4) ** 2 + (y - 5) ** 2))
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def G1(x, y):
    """G1 = 9/d1 + 12/d2 + 25/d3 with centers (2,4), (5,2), (4,5)."""
    return (9 / np.sqrt((x - 2) ** 2 + (y - 4) ** 2)
            + 12 / np.sqrt((x - 5) ** 2 + (y - 2) ** 2)
            + 25 / np.sqrt((x - 4) ** 2 + (y - 5) ** 2))
```
</details>

In [ ]:
# 🧪 Self-check for Problem 3.4
v34 = G1(4.0, 4.9)
assert v34 is not None, "❌ G1 returned None — replace the 'pass'. See Hint 2 in Part 3."
v34 = float(v34)
assert np.isfinite(v34) and v34 > 100.0, "❌ G1(4, 4.9) should be large: the 25/distance term alone is 25/0.1 = 250. Check the weight and center of the third term."
far34 = float(G1(-1.0, -1.0))
assert np.isfinite(far34) and 0.0 < far34 < 10.0, "❌ G1(-1, -1) should be a small positive number (all three inverse distances are large there)."
assert float(G1(4.0, 4.9)) > float(G1(4.0, 4.5)) > float(G1(4.0, 3.0)), "❌ G1 must decrease as you move away from the (4, 5) center along x = 4."
assert float(G1(2.0, 4.2)) > 30.0, "❌ Near the (2, 4) center the 9/distance term dominates: 9/0.2 = 45. Check the first term."
rng_g1 = np.random.default_rng(0)
Ag, Bg = rng_g1.normal(3, 1, size=(7, 5)), rng_g1.normal(2, 1, size=(7, 5))
Zg34 = G1(Ag, Bg)
assert getattr(Zg34, "shape", None) == (7, 5) and np.isfinite(Zg34).all(), "❌ G1 must work element-wise on equal-shape arrays (only points exactly at a center are non-finite)."
print("✅ Problem 3.4 passed — G1 implemented.")

## 🎯 Problem 3.5 — Mask the singularity and locate the extrema

**Given:** a grid function `func`, meshgrid arrays `X, Y` and a `cap`.

**Required:** write `mask_and_locate(func, X, Y, cap=200.0)` that:

1. evaluates `func` on the grid (silence divide warnings with `np.errstate`),
2. builds a masked array dropping every cell that is **non-finite or above `cap`** (the singular cores and their immediate neighborhood),
3. returns a dict with keys `'masked'` (the masked array), `'min_point'` and `'max_point'` (each an `(x, y)` tuple on the grid).

**Expected output:** for `G1` on $[-1, 8]^2$ with cap 200: cells masked near the three centers, the max point in the near-field ring around $(4, 5)$, and the min point at the far corner $(-1, -1)$.

In [ ]:
def mask_and_locate(func, X, Y, cap=200.0):
    """Evaluate func on (X, Y), mask non-finite / > cap cells, locate extrema.

    Args:
        func: element-wise function of two arrays
        X, Y: meshgrid arrays
        cap: values above cap are masked (singular near-field)

    Returns:
        dict with keys 'masked', 'min_point', 'max_point'
    """
    # TODO: Your code here
    pass

# Demo call (G1 on the Part 1 grid)
demo_rep = mask_and_locate(G1, X, Y)
if demo_rep is not None:
    print("masked cells:", int(demo_rep["masked"].mask.sum()))
    print("min at", demo_rep["min_point"], "| max at", demo_rep["max_point"])
else:
    print("Implement mask_and_locate to power this demo.")

<details>
<summary>💡 Hint 1 — mask two things</summary>

Mask cells that are non-finite AND cells above the cap, combined with `|`. On a masked array, `np.ma.argmin`/`np.ma.argmax` ignore masked cells. Wrap the evaluation in `np.errstate(divide="ignore", invalid="ignore")` to keep the log clean.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
with np.errstate(divide="ignore", invalid="ignore"):
    Z = func(X, Y)
Zm = np.ma.masked_where(~np.isfinite(Z) | (Z > cap), Z)
r, c = np.unravel_index(np.ma.argmin(Zm), Zm.shape)
# same for argmax; build the dict with (X[r, c], Y[r, c]) tuples
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def mask_and_locate(func, X, Y, cap=200.0):
    """Evaluate func on (X, Y), mask non-finite / > cap cells, locate extrema."""
    with np.errstate(divide="ignore", invalid="ignore"):
        Z = func(X, Y)
    Zm = np.ma.masked_where(~np.isfinite(Z) | (Z > cap), Z)
    r_min, c_min = np.unravel_index(np.ma.argmin(Zm), Zm.shape)
    r_max, c_max = np.unravel_index(np.ma.argmax(Zm), Zm.shape)
    return {"masked": Zm,
            "min_point": (float(X[r_min, c_min]), float(Y[r_min, c_min])),
            "max_point": (float(X[r_max, c_max]), float(Y[r_max, c_max]))}
```
</details>

In [ ]:
# 🧪 Self-check for Problem 3.5
grid5 = evaluate_on_grid(lambda a, b: a + b, n=20)
assert grid5 is not None, "❌ evaluate_on_grid returned None — finish Problem 3.1 first."
Xg5, Yg5, _ = grid5
rep5 = mask_and_locate(G1, Xg5, Yg5, cap=200.0)
assert rep5 is not None, "❌ mask_and_locate returned None — replace the 'pass'. See Hint 2 in Part 3."
assert set(rep5.keys()) >= {"masked", "min_point", "max_point"}, f"❌ Dict needs keys 'masked', 'min_point', 'max_point' — got {sorted(rep5.keys())}."
Zm5 = rep5["masked"]
assert getattr(Zm5, "mask", None) is not None, "❌ 'masked' must be a masked numpy array (np.ma.masked_where)."
# (a coarse 20x20 grid may not land inside a singular core — the fine-grid test below is the real one)
vals5 = Zm5.compressed()
assert np.isfinite(vals5).all() and vals5.max() <= 200.0 + 1e-9, "❌ Every unmasked value must be finite and at most cap."
assert vals5.max() > 100.0, "❌ The largest unmasked value should sit just under the cap in the near-field ring of (4, 5) — check the mask condition."
assert float(Zm5.min()) < 10.0, "❌ The masked grid minimum should be small — it lives at the far corner."
min_pt5 = rep5["min_point"]
assert np.hypot(min_pt5[0] + 1.0, min_pt5[1] + 1.0) < 0.6, "❌ The masked grid minimum of G1 sits at the far corner (-1, -1) — check the row/col -> y/x mapping in your locations."
grid5f = evaluate_on_grid(lambda a, b: a + b, n=150)
Xg5f, Yg5f, _ = grid5f
rep5f = mask_and_locate(G1, Xg5f, Yg5f, cap=200.0)
assert int(np.asarray(rep5f["masked"].mask).sum()) > 0, "❌ On a fine (150) grid the singular cores of G1 must be masked."
print("✅ Problem 3.5 passed — singularities masked, extrema found.")

## Part 4: G2 — the convex bowl & mini-challenge (≈10 min)

The assignment's third function flips the fraction — **distances divided by the weights**:

$$G_2(x, y) = \frac{r_1}{9} + \frac{r_2}{12} + \frac{r_3}{25}$$

- Finite everywhere (distances never blow up) — no masking needed.
- A sum of distances is **convex** (bowl-shaped): one interior minimum near the weighted Fermat point (pulled toward the *smallest* weight), and the maximum at the farthest corner of the domain.
- `np.argmin` / `np.argmax` on the grid values locate both numerically.

**Mini-challenge:** one figure with all three functions, extrema marked — the exact deliverable of Lab Assignment 03.

In [ ]:
# --- Worked example: G2 needs no mask ---
def G2_example(x, y):
    return (np.sqrt((x - 2) ** 2 + (y - 4) ** 2) / 9
            + np.sqrt((x - 5) ** 2 + (y - 2) ** 2) / 12
            + np.sqrt((x - 4) ** 2 + (y - 5) ** 2) / 25)

Z_g2 = G2_example(X, Y)
print("G2 finite everywhere:", bool(np.isfinite(Z_g2).all()))
imin2 = np.unravel_index(Z_g2.argmin(), Z_g2.shape)
imax2 = np.unravel_index(Z_g2.argmax(), Z_g2.shape)
print("G2 grid minimum at (x, y) =", (x[imin2[1]], y[imin2[0]]), " <- interior point")
print("G2 grid maximum at (x, y) =", (x[imax2[1]], y[imax2[0]]), " <- farthest corner")

## 🎯 Problem 3.6 — Implement G2 and verify its bowl shape

**Given:** the same three centers and weights as $G_1$.

**Required:** write `G2(x, y)` computing $\frac{r_1}{9} + \frac{r_2}{12} + \frac{r_3}{25}$ element-wise.

**Expected output:** a convex, everywhere-finite bowl. The self-check verifies convexity directly: the value at the midpoint of two points never exceeds the average of the endpoint values.

In [ ]:
def G2(x, y):
    """G2(x, y) = sqrt((x-2)^2+(y-4)^2)/9 + sqrt((x-5)^2+(y-2)^2)/12
                  + sqrt((x-4)^2+(y-5)^2)/25, element-wise."""
    # TODO: Your code here
    pass

# Demo call
demo_w = G2(4.0, 4.0)
if demo_w is not None:
    print("G2(4.0, 4.0) =", demo_w)
    print("G2(8.0, 8.0) =", G2(8.0, 8.0))
else:
    print("Implement G2 to power this demo.")

<details>
<summary>💡 Hint 1 — flip the fraction</summary>

Same centers, same weights — but this time each term is the distance DIVIDED by its weight. `np.sqrt` first, divide by the weight after.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
return (np.sqrt((x - 2) ** 2 + (y - 4) ** 2) / 9
        + np.sqrt((x - 5) ** 2 + (y - 2) ** 2) / 12
        + np.sqrt((x - 4) ** 2 + (y - 5) ** 2) / 25)
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def G2(x, y):
    """G2 = d1/9 + d2/12 + d3/25 with centers (2,4), (5,2), (4,5)."""
    return (np.sqrt((x - 2) ** 2 + (y - 4) ** 2) / 9
            + np.sqrt((x - 5) ** 2 + (y - 2) ** 2) / 12
            + np.sqrt((x - 4) ** 2 + (y - 5) ** 2) / 25)
```
</details>

In [ ]:
# 🧪 Self-check for Problem 3.6
w36 = G2(4.0, 4.0)
assert w36 is not None, "❌ G2 returned None — replace the 'pass'. See Hint 2 in Part 4."
assert np.isfinite(float(w36)) and float(w36) > 0.0, "❌ G2 must be positive and finite everywhere."
mid36 = float(G2(4.0, 4.0)); avg36 = 0.5 * (float(G2(0.0, 0.0)) + float(G2(8.0, 8.0)))
assert mid36 <= avg36 + 1e-12, "❌ A sum of distances is convex — the midpoint value must not exceed the endpoint average. Check the square roots."
res6 = evaluate_on_grid(G2, n=150)
assert res6 is not None, "❌ evaluate_on_grid returned None — finish Problem 3.1 first."
Xg6, Yg6, Zg6 = res6
imin6 = np.unravel_index(Zg6.argmin(), Zg6.shape)
imax6 = np.unravel_index(Zg6.argmax(), Zg6.shape)
assert imin6[0] not in (0, Zg6.shape[0] - 1) and imin6[1] not in (0, Zg6.shape[1] - 1), "❌ The G2 minimum is an interior point — a convex bowl's minimum is not on the grid border."
assert np.hypot(Xg6[imax6] - 8.0, Yg6[imax6] - 8.0) < 0.1, "❌ The G2 maximum sits at the farthest corner (8, 8)."
print("✅ Problem 3.6 passed — G2 is a well-behaved bowl.")

## 🎯 Problem 3.7 — Mini-challenge: the full assignment figure

**Given:** your `F`, `G1`, `G2`, `evaluate_on_grid`, `mask_and_locate` and `plot_contour` from this lab.

**Required:** write `plot_all_three(n=200)` producing **one figure with three side-by-side contour plots** — $F$, masked $G_1$ (cap 200), $G_2$ — each with its grid minimum marked by a white `'x'` and its grid maximum by a white `'*'`, each with a title. **Returns the `Figure`.**

Lay it out with `fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))` and draw each panel by hand (contourf + lines + colorbar + markers) — the three panels need different handling ($G_1$ must be masked), so `plot_contour` alone is not enough.

**This is exactly what Lab Assignment 03 asks you to deliver** — three functions, three contour plots, extrema marked. Finishing this problem means the assignment is assembly, not discovery.

In [ ]:
def plot_all_three(n=200):
    """One figure, three panels: F, masked G1, G2 — extrema marked on each.

    Args:
        n: grid points per axis

    Returns:
        matplotlib Figure with 1 x 3 contour panels
    """
    # TODO: Your code here
    pass

# Demo call
fig_all = plot_all_three()
if fig_all is not None:
    print("plot_all_three returned a", type(fig_all).__name__)
    plt.show()
else:
    print("Implement plot_all_three to power this demo.")

<details>
<summary>💡 Hint 1 — one loop, three panels</summary>

Loop over `(func, title, cap)` triples. Inside: evaluate (errstate), mask if a cap is given, `np.ma.asarray` so argmin/argmax work for both cases, contourf + lines + colorbar, locate extrema with unravel_index, plot the two markers, set the title.
</details>

<details>
<summary>💡 Hint 2 — the approach</summary>

```text
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (func, title, cap) in zip(axes, specs):
    evaluate -> mask if cap -> Zm = np.ma.asarray(Z)
    contourf + contour lines + colorbar
    r, c = unravel_index(ma.argmin(Zm)); ax.plot(x[c], y[r], "wx")
    r, c = unravel_index(ma.argmax(Zm)); ax.plot(x[c], y[r], "w*")
    ax.set_title(title)
fig.tight_layout(); return fig
```
</details>

<details>
<summary>✅ Reveal solution</summary>

```python
def plot_all_three(n=200):
    """One figure, three panels: F, masked G1, G2 — extrema marked on each."""
    x = np.linspace(-1, 8, n); y = np.linspace(-1, 8, n)
    X, Y = np.meshgrid(x, y)
    specs = [(F, "F(x,y) = sin^2(x) + cos^2(y)", None),
             (G1, "G1(x,y): weighted inverse distances", 200.0),
             (G2, "G2(x,y): weighted distances", None)]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, (func, title, cap) in zip(axes, specs):
        with np.errstate(divide="ignore", invalid="ignore"):
            Z = func(X, Y)
        if cap is not None:
            Z = np.ma.masked_where(~np.isfinite(Z) | (Z > cap), Z)
        Zm = np.ma.asarray(Z)
        cs = ax.contourf(X, Y, Zm, levels=15, cmap="viridis")
        ax.contour(X, Y, Zm, levels=15, colors="k", linewidths=0.3, alpha=0.4)
        fig.colorbar(cs, ax=ax)
        r_min, c_min = np.unravel_index(np.ma.argmin(Zm), Zm.shape)
        r_max, c_max = np.unravel_index(np.ma.argmax(Zm), Zm.shape)
        ax.plot(x[c_min], y[r_min], "wx", markersize=11, markeredgewidth=2.5)
        ax.plot(x[c_max], y[r_max], "w*", markersize=13)
        ax.set_title(title)
    fig.tight_layout()
    return fig
```
</details>

In [ ]:
# 🧪 Self-check for Problem 3.7
figA = plot_all_three()
assert figA is not None, "❌ plot_all_three returned None — replace the 'pass'. See Hint 2 in Part 4."
import matplotlib.figure as _mpl_figure2
import matplotlib.collections as _mpl_coll2
from matplotlib.contour import QuadContourSet as _QCS2

def _has_contourf2(ax):
    # matplotlib >= 3.8 registers contourf as a QuadContourSet in ax.collections;
    # older versions register a QuadMesh. Accept either.
    return any(isinstance(c, (_mpl_coll2.QuadMesh, _QCS2)) for c in ax.collections)

assert isinstance(figA, _mpl_figure2.Figure), "❌ Return the matplotlib Figure itself."
axesA = figA.get_axes()[:3]  # the 3 contour panels (their colorbars come after)
assert len(axesA) == 3, "❌ Expected 3 contour panels."
for i, ax in enumerate(axesA):
    assert _has_contourf2(ax), f"❌ Panel {i} has no filled contour (ax.contourf)."
    assert len(ax.get_lines()) == 2, f"❌ Panel {i} should carry exactly 2 markers (min 'x' and max '*')."
    assert ax.get_title().strip(), f"❌ Panel {i} needs a title."
assert axesA[0].get_title() != axesA[1].get_title(), "❌ The three panels should have different titles."
print("✅ Problem 3.7 passed — the full assignment figure works. Lab Assignment 03 awaits!")

## 🎉 You've completed Lab Activity 3

You have mastered:
- Building evaluation grids with `np.meshgrid` (mind the row/col → y/x mapping) and evaluating functions element-wise
- Drawing filled contours with `ax.contourf` + line overlays + colorbar, and marking extrema with `'x'` / `'*'`
- Reading local minima and maxima off a contour map — analytically for $F$, numerically with `argmin`/`argmax` for $G_1$ and $G_2$
- Taming singularities: `np.errstate` + `np.ma.masked_where` cap the near-field blow-up of $G_1$
- Assembling all three assignment functions into one annotated figure

**You are now ready for Lab Assignment 03 on the course portal — the assignment asks for the same techniques without hints.**

💡 **Tip:** restart the kernel and run every cell top-to-bottom once more — each 🧪 self-check should print ✅, and the row/col vs x/y mapping is worth making second nature.